In [1]:
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns
import random
import math

from sklearn.model_selection import train_test_split, RandomizedSearchCV, StratifiedKFold
from sklearn.metrics import confusion_matrix, classification_report, roc_curve, auc

from scikeras.wrappers import KerasClassifier

from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras.preprocessing.image import ImageDataGenerator

from core_kfold import train_kfold, load_dataset

BATCH_SIZE = 16

tf.keras.mixed_precision.set_global_policy('mixed_float16')
print("Mixed Precision (float16) đã bật")

gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print(f"Memory growth enabled cho {len(gpus)} GPU")
else:
    print("Không tìm thấy GPU, đang chạy trên CPU")

print("Import thành công!")

d:\App\python\Lib\site-packages\keras\src\export\tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


Mixed Precision (float16) đã bật
Không tìm thấy GPU, đang chạy trên CPU
Import thành công!


In [2]:
DATA_DIR = "D:/Do_An/Data_Final/processed_dataset"

X, y = load_dataset(DATA_DIR)
print("Total samples:", len(X))

X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print("Train_val:", len(X_trainval))
print("Test:", len(X_test))

Total samples: 2810
Train_val: 2248
Test: 562


In [3]:
from tensorflow.keras import layers, regularizers

def build_model(dense_units=128, dropout_rate=0.5, unfreeze_ratio=0.25):
    base_model = MobileNetV2(
        weights='imagenet',
        include_top=False,
        input_shape=(224, 224, 3)
    )
    base_model.trainable = False

    x = base_model.output
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dense(
        dense_units,
        activation='relu',
        kernel_regularizer=regularizers.l2(1e-4)
    )(x)
    x = layers.Dropout(dropout_rate)(x)
    outputs = layers.Dense(1, activation='sigmoid')(x)

    model = tf.keras.Model(base_model.input, outputs)
    return model, base_model

In [4]:
print("\nĐang thực hiện Hyperparameter Tuning cho MobileNetV2")

# Giảm data để tuning nhanh
X_small, _, y_small, _ = train_test_split(
    X_trainval, y_trainval, test_size=0.5, stratify=y_trainval, random_state=42
)

X_train_tune = []
for path in X_small:
    img = tf.keras.preprocessing.image.load_img(path, target_size=(224, 224))
    img = tf.keras.preprocessing.image.img_to_array(img)
    X_train_tune.append(img)

X_train_tune = np.array(X_train_tune)
X_train_tune = preprocess_input(X_train_tune)

print("Tuning data shape:", X_train_tune.shape)

def create_model(dense_units=128, dropout_rate=0.5, unfreeze_ratio=0.25):
    model, _ = build_model(dense_units=dense_units, 
                           dropout_rate=dropout_rate, 
                           unfreeze_ratio=unfreeze_ratio)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-4),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return model

clf = KerasClassifier(model=create_model, epochs=8, batch_size=BATCH_SIZE, verbose=0)

param_dist = {
    'model__dense_units': [64, 128, 256],
    'model__dropout_rate': [0.4, 0.5, 0.6],
    'model__unfreeze_ratio': [0.15, 0.20, 0.25, 0.30]
}

search = RandomizedSearchCV(
    clf,
    param_distributions=param_dist,
    n_iter=5,
    cv=StratifiedKFold(n_splits=3, shuffle=True, random_state=42),
    verbose=2,
    n_jobs=1
)

search.fit(X_train_tune, y_small)
best_params = search.best_params_

print("\nBest parameters found:", best_params)


Đang thực hiện Hyperparameter Tuning
Tuning data shape: (1124, 224, 224, 3)
Fitting 3 folds for each of 5 candidates, totalling 15 fits


KeyboardInterrupt: 

In [ ]:
print("\nTrain K-Fold với Best Parameters")

# Lấy best params
best_dense = best_params['model__dense_units']
best_dropout = best_params['model__dropout_rate']
best_unfreeze = best_params['model__unfreeze_ratio']

def build_best_model():
    model, base = build_model(
        dense_units=best_dense,
        dropout_rate=best_dropout,
        unfreeze_ratio=best_unfreeze
    )
    return model, base

def unfreeze_best(base_model):
    total = len(base_model.layers)
    start = int(total * (1 - best_unfreeze))
    for i, layer in enumerate(base_model.layers):
        layer.trainable = i >= start
    print(f"[MobileNetV2] Unfrozen {best_unfreeze:.0%} layers")

build_best_model.unfreeze = unfreeze_best

result = train_kfold(
    build_model_fn=build_best_model,
    preprocess_input=preprocess_input,
    model_name="MobileNetV2",
    X=X_trainval,
    y=y_trainval
)

print(f"\nHoàn thành! Best model: {result['best_model_path']}")

In [ ]:
print("\n=== Đánh giá trên Test Set ===")

best_model = tf.keras.models.load_model(result['best_model_path'])

test_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)
test_gen = MeatDataSequence(X_test, y_test, preprocess_input, augment=False, batch_size=BATCH_SIZE)

y_probs = best_model.predict(test_gen, steps=math.ceil(len(X_test)/BATCH_SIZE), verbose=0)
y_pred = (y_probs > 0.5).astype(int).flatten()

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Fresh', 'Rotten']))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Fresh', 'Rotten'], yticklabels=['Fresh', 'Rotten'])
plt.title('Confusion Matrix')
plt.show()

# ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_probs)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(8,6))
plt.plot(fpr, tpr, label=f'ROC Curve (AUC = {roc_auc:.4f})', linewidth=2)
plt.plot([0, 1], [0, 1], 'k--')
plt.title('ROC Curve - Test Set')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.legend()
plt.grid(True)
plt.show()

print(f"AUC Score: {roc_auc:.4f}")

# Wrong Predictions
wrong_images = [(X_test[i], y_test[i], y_pred[i]) for i in range(len(X_test)) if y_pred[i] != y_test[i]]
print(f"\nSố ảnh dự đoán sai: {len(wrong_images)}")

if wrong_images:
    plt.figure(figsize=(12, 8))
    samples = random.sample(wrong_images, min(6, len(wrong_images)))
    label_map = {0: "Fresh", 1: "Rotten"}
    for i, (path, true, pred) in enumerate(samples):
        img = tf.keras.preprocessing.image.load_img(path)
        plt.subplot(2, 3, i+1)
        plt.imshow(img)
        plt.title(f"True: {label_map[true]}\nPred: {label_map[pred]}")
        plt.axis("off")
    plt.suptitle("Wrong Predictions")
    plt.show()

Fitting 3 folds for each of 6 candidates, totalling 18 fits
[CV] END model__dense_units=32, model__dropout_rate=0.6, model__learning_rate=1e-05; total time= 1.8min
[CV] END model__dense_units=32, model__dropout_rate=0.6, model__learning_rate=1e-05; total time= 1.8min
[CV] END model__dense_units=32, model__dropout_rate=0.6, model__learning_rate=1e-05; total time= 1.8min
[CV] END model__dense_units=64, model__dropout_rate=0.5, model__learning_rate=5e-05; total time= 1.8min
[CV] END model__dense_units=64, model__dropout_rate=0.5, model__learning_rate=5e-05; total time= 1.8min
[CV] END model__dense_units=64, model__dropout_rate=0.5, model__learning_rate=5e-05; total time= 2.2min
[CV] END model__dense_units=64, model__dropout_rate=0.4, model__learning_rate=0.0001; total time= 2.0min
[CV] END model__dense_units=64, model__dropout_rate=0.4, model__learning_rate=0.0001; total time= 1.9min
[CV] END model__dense_units=64, model__dropout_rate=0.4, model__learning_rate=0.0001; total time= 1.9min
[

KeyboardInterrupt: 

In [ ]:
def build_best_model():
    params = search.best_params_

    model, base = build_model(
        learning_rate=params['model__learning_rate'],
        dropout_rate=params['model__dropout_rate'],
        dense_units=params['model__dense_units']
    )

    base.trainable = False  # 🔥 KHÓA LUÔN

    return model, base

In [ ]:
result = train_kfold(
    build_model_fn=build_best_model,
    preprocess_input=preprocess_input,
    model_name="MobileNetV2",
    X=X_trainval,
    y=y_trainval
)

print(f"\nHoàn thành! Best model được lưu tại: {result['best_model_path']}")
print(f"Tổng thời gian huấn luyện: {result['total_training_time']/60:.2f} phút")

In [ ]:
plt.figure()

for i, hist in enumerate(result["histories"]):
    plt.plot(hist["val_loss"], label=f"Fold {i+1}")

plt.title("Validation Loss per Fold")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.show()

In [ ]:
accs = result["accs"]

plt.figure()
plt.bar(range(1, len(accs)+1), accs)

plt.title("Accuracy per Fold")
plt.xlabel("Fold")
plt.ylabel("Accuracy")

for i, v in enumerate(accs):
    plt.text(i+1, v, f"{v:.3f}", ha='center')

plt.show()

In [ ]:
final_model = tf.keras.models.load_model(result["best_model_path"])

test_seq = MeatDataSequence(
    X_test,
    y_test,
    preprocess_input,   
    augment=False,
    batch_size=16
)

y_probs = final_model.predict(test_seq, verbose=1)
y_preds = (y_probs > 0.5).astype(int).flatten()

In [ ]:
print("\nCLASSIFICATION REPORT")
print(classification_report(y_test, y_preds, target_names=["Fresh", "Rotten"]))

In [ ]:
cm = confusion_matrix(y_test, y_preds)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", 
            xticklabels=["Fresh", "Rotten"], yticklabels=["Fresh", "Rotten"])
plt.title("Confusion Matrix (Test Set)")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

In [ ]:
wrong_images = []

for i in range(len(X_test)):
    if y_preds[i] != y_test[i]:
        wrong_images.append((X_test[i], y_test[i], y_preds[i]))

print("Số ảnh dự đoán sai:", len(wrong_images))

plt.figure(figsize=(12,8))

samples = random.sample(wrong_images, min(6, len(wrong_images)))
label_map = {0: "fresh", 1: "rotten"}

for i, (path, true, pred) in enumerate(samples):
    img = tf.keras.preprocessing.image.load_img(path)

    plt.subplot(2,3,i+1)
    plt.imshow(img)
    plt.title(f"True: {label_map[true]} | Pred: {label_map[pred]}")
    plt.axis("off")

plt.suptitle("Wrong Predictions")
plt.show()

In [ ]:
y_probs = final_model.predict(test_seq).flatten()

fpr, tpr, _ = roc_curve(y_test, y_probs)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(8,6))
plt.plot(fpr, tpr, label=f"AUC = {roc_auc:.4f}")
plt.plot([0,1], [0,1], linestyle='--')

plt.xlabel("FPR")
plt.ylabel("TPR")
plt.title("ROC Curve")
plt.legend()
plt.grid()
plt.show()

print("AUC:", roc_auc)